# Caught & Shared — Virtual Proxy Items (Node Transformation)
**Preserving strict bipartite structure while enabling mentor influence**

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch_geometric.data import HeteroData
import matplotlib.pyplot as plt

np.random.seed(42)
torch.manual_seed(42)

#### 1. Synthetic Data + Proxy Items Creation

In [ ]:
num_real_users = 20
num_real_items = 60

interactions = []
for u in range(num_real_users):
    num_inter = np.random.randint(4, 12)
    items = np.random.choice(num_real_items, num_inter, replace=False)
    for i in items:
        interactions.append((u, i))

inter_df = pd.DataFrame(interactions, columns=['user_id', 'item_id'])

mentor_assignments = [
    (0, 5, 0.75),   # User 0 → Mentor 5 with strength 0.75
    (0, 7, 0.45),   # User 0 also follows Mentor 7
    (1, 12, 0.85),
    (2, 5, 0.65),
    (3, 8, 0.90),
    (4, 15, 0.70),
]

print(f"{len(inter_df)} real user-item interactions")
print(f"{len(mentor_assignments)} mentor assignments")

#### 2. Creating Virtual Proxy Items

In [ ]:
unique_mentors = list(set([m for _, m, _ in mentor_assignments]))
num_proxy_items = len(unique_mentors)
proxy_id_offset = num_real_items

mentor_to_proxy = {mentor_id: proxy_id_offset + i 
                   for i, mentor_id in enumerate(unique_mentors)}

proxy_interactions = []
for user_id, mentor_id, alpha in mentor_assignments:
    proxy_item_id = mentor_to_proxy[mentor_id]
    proxy_interactions.append((user_id, proxy_item_id, alpha))  # alpha as interaction strength

proxy_df = pd.DataFrame(proxy_interactions, columns=['user_id', 'proxy_item_id', 'alpha'])

#### 3. Building Bipartite Graph (with Proxy Items)

In [ ]:
data = HeteroData()

total_items = num_real_items + num_proxy_items

data['user'].x = torch.randn(num_real_users, 16)
data['item'].x = torch.randn(total_items, 16)

real_src = torch.tensor(inter_df['user_id'].values)
real_dst = torch.tensor(inter_df['item_id'].values)
data['user', 'interact', 'item'].edge_index = torch.stack([real_src, real_dst])

proxy_src = torch.tensor(proxy_df['user_id'].values)
proxy_dst = torch.tensor(proxy_df['proxy_item_id'].values)
data['user', 'interact', 'item'].edge_index = torch.cat([
    data['user', 'interact', 'item'].edge_index,
    torch.stack([proxy_src, proxy_dst])
], dim=1)

# Optional: add edge weights for proxy interactions
data['user', 'interact', 'item'].edge_weight = torch.cat([
    torch.ones(len(real_src)), 
    torch.tensor(proxy_df['alpha'].values, dtype=torch.float)
])

print(data)